# Train Number Landmark Model for SmartSignLanguage

Notebook này dùng để train riêng model nhận diện số `0-9` theo chuẩn inference hiện tại của project.

Khác với model CNN ảnh trực tiếp, project SmartSignLanguage đang nhận diện bằng chuỗi landmark:

- Input model: `(SEQ_LEN, 225)`
- 225 feature = left hand `63` + right hand `63` + pose `99`
- Output: 10 lớp số `0-9`

Sau khi train, notebook xuất:

- `ai-model/model/model_number.keras`
- `ai-model/model/mapping_number.json`

Lưu ý: inference server hiện tại đang tự load `model_alnum.keras` cho mode `alnum`. Nếu muốn dùng riêng model số này trong app, cần thêm một mode mới như `number` trong `ai-model/inference/main.py`, hoặc đổi output path thành `model_alnum.keras` nếu chỉ muốn mode alnum nhận diện số.

## 1. Setup

In [ ]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)
print('TensorFlow:', tf.__version__)


## 2. Configuration

`NUMBER_DATASET_DIR` cần trỏ tới thư mục có cấu trúc:

```text
dataset/
  0/
  1/
  2/
  ...
  9/
```

Mỗi folder chứa ảnh của ký hiệu số tương ứng.

In [ ]:
# Change this path when running on Kaggle/local.
NUMBER_DATASET_DIR = Path('/kaggle/input/your-asl-number-dataset')

PROJECT_MODEL_DIR = Path('../model')
HAND_TASK_PATH = PROJECT_MODEL_DIR / 'hand_landmarker.task'
POSE_TASK_PATH = PROJECT_MODEL_DIR / 'pose_landmarker.task'

OUTPUT_MODEL_PATH = PROJECT_MODEL_DIR / 'model_number.keras'
OUTPUT_MAPPING_PATH = PROJECT_MODEL_DIR / 'mapping_number.json'
CACHE_PATH = Path('./alphabet/number_landmarks_cache.npz')

IMAGE_SIZE = 224
SEQ_LEN = 5
FEATURE_DIM = 225
BATCH_SIZE = 256
EPOCHS = 35
MAX_IMAGES_PER_CLASS = None  # Example: 2500 for quick experiments.

NUMBER_LABELS = [str(i) for i in range(10)]

print('Number dataset:', NUMBER_DATASET_DIR)
print('Output model:', OUTPUT_MODEL_PATH)
print('Output mapping:', OUTPUT_MAPPING_PATH)


## 3. Import MediaPipe

In [ ]:
try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    print('MediaPipe:', mp.__version__)
except Exception as exc:
    raise RuntimeError('Install MediaPipe first: pip install mediapipe') from exc


## 4. Load Image Paths

In [ ]:
def list_images_for_label(root: Path, label: str):
    folder = root / label
    if not folder.exists():
        return []
    paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
        paths.extend(folder.glob(ext))
    return sorted(paths)

image_paths = []
string_labels = []

for label in NUMBER_LABELS:
    paths = list_images_for_label(NUMBER_DATASET_DIR, label)
    if MAX_IMAGES_PER_CLASS:
        paths = paths[:MAX_IMAGES_PER_CLASS]
    image_paths.extend(paths)
    string_labels.extend([label] * len(paths))

image_paths = np.array(image_paths, dtype=object)
string_labels = np.array(string_labels, dtype=str)

selected_labels = [label for label in NUMBER_LABELS if np.any(string_labels == label)]
label_to_index = {label: index for index, label in enumerate(selected_labels)}
index_to_label = {index: label for label, index in label_to_index.items()}
int_labels = np.array([label_to_index[label] for label in string_labels], dtype=np.int64)

print('Images:', len(image_paths))
print('Classes:', selected_labels)
if len(selected_labels) != 10:
    print('Warning: Not all number classes 0-9 were found.')


## 5. Landmark Extraction

In [ ]:
def create_detectors(hand_task_path: Path, pose_task_path: Path):
    if not hand_task_path.exists():
        raise FileNotFoundError(f'Missing hand task file: {hand_task_path}')
    if not pose_task_path.exists():
        raise FileNotFoundError(f'Missing pose task file: {pose_task_path}')

    hand_detector = mp_vision.HandLandmarker.create_from_options(
        mp_vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(hand_task_path)),
            num_hands=2,
            min_hand_detection_confidence=0.3,
            min_hand_presence_confidence=0.3,
            min_tracking_confidence=0.3,
            running_mode=mp_vision.RunningMode.IMAGE,
        )
    )
    pose_detector = mp_vision.PoseLandmarker.create_from_options(
        mp_vision.PoseLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(pose_task_path)),
            min_pose_detection_confidence=0.3,
            min_tracking_confidence=0.3,
            running_mode=mp_vision.RunningMode.IMAGE,
        )
    )
    return hand_detector, pose_detector

def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    seq = seq.copy()
    for frame_index in range(seq.shape[0]):
        left_hand = seq[frame_index, 0:63].reshape(21, 3)
        if np.any(left_hand != 0):
            left_hand = left_hand - left_hand[0]
            scale = np.max(np.linalg.norm(left_hand, axis=1)) + 1e-8
            seq[frame_index, 0:63] = (left_hand / scale).flatten()

        right_hand = seq[frame_index, 63:126].reshape(21, 3)
        if np.any(right_hand != 0):
            right_hand = right_hand - right_hand[0]
            scale = np.max(np.linalg.norm(right_hand, axis=1)) + 1e-8
            seq[frame_index, 63:126] = (right_hand / scale).flatten()

        pose = seq[frame_index, 126:225].reshape(33, 3)
        if np.any(pose != 0):
            pose = pose - pose[0]
            shoulder_width = np.linalg.norm(pose[11] - pose[12]) + 1e-8
            seq[frame_index, 126:225] = (pose / shoulder_width).flatten()
    return seq.astype(np.float32)

def extract_features_from_image(image_path: Path, hand_detector, pose_detector):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None
    image_bgr = cv2.resize(image_bgr, (IMAGE_SIZE, IMAGE_SIZE))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    left_hand = np.zeros(63, dtype=np.float32)
    right_hand = np.zeros(63, dtype=np.float32)

    hand_result = hand_detector.detect(mp_image)
    for index, handedness in enumerate(hand_result.handedness):
        if index >= len(hand_result.hand_landmarks):
            continue
        hand_label = handedness[0].category_name
        hand_points = hand_result.hand_landmarks[index]
        flat_points = np.array(
            [[landmark.x, landmark.y, landmark.z] for landmark in hand_points],
            dtype=np.float32,
        ).flatten()
        if hand_label == 'Left':
            left_hand = flat_points
        else:
            right_hand = flat_points

    if not np.any(left_hand != 0) and not np.any(right_hand != 0):
        return None

    pose = np.zeros(99, dtype=np.float32)
    pose_result = pose_detector.detect(mp_image)
    if pose_result.pose_landmarks:
        pose = np.array(
            [[landmark.x, landmark.y, landmark.z] for landmark in pose_result.pose_landmarks[0]],
            dtype=np.float32,
        ).flatten()

    features = np.concatenate([left_hand, right_hand, pose]).astype(np.float32)
    sequence = np.repeat(features[None, :], SEQ_LEN, axis=0)
    return normalize_sequence(sequence)


## 6. Build or Load Cache

In [ ]:
def build_landmark_cache(image_paths, int_labels):
    hand_detector, pose_detector = create_detectors(HAND_TASK_PATH, POSE_TASK_PATH)
    X = []
    y = []
    skipped = 0

    for index, (image_path, label) in enumerate(zip(image_paths, int_labels)):
        sequence = extract_features_from_image(Path(image_path), hand_detector, pose_detector)
        if sequence is None:
            skipped += 1
        else:
            X.append(sequence)
            y.append(label)

        if (index + 1) % 1000 == 0:
            print(f'Processed {index + 1}/{len(image_paths)} | usable={len(X)} | skipped={skipped}')

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), skipped

CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH, allow_pickle=True)
    X = cache['X'].astype(np.float32)
    y = cache['y'].astype(np.int64)
    selected_labels = cache['selected_labels'].tolist()
    label_to_index = {label: index for index, label in enumerate(selected_labels)}
    index_to_label = {index: label for label, index in label_to_index.items()}
    print('Loaded cache:', CACHE_PATH)
else:
    X, y, skipped = build_landmark_cache(image_paths, int_labels)
    np.savez_compressed(
        CACHE_PATH,
        X=X,
        y=y,
        selected_labels=np.array(selected_labels, dtype=object),
    )
    print('Saved cache:', CACHE_PATH, 'skipped:', skipped)

print('X:', X.shape)
print('y:', y.shape)


## 7. Train/Validation/Test Split

In [ ]:
indices = np.arange(len(X))
np.random.shuffle(indices)

train_end = int(len(indices) * 0.8)
valid_end = int(len(indices) * 0.9)

train_idx = indices[:train_end]
valid_idx = indices[train_end:valid_end]
test_idx = indices[valid_end:]

X_train, y_train = X[train_idx], y[train_idx]
X_valid, y_valid = X[valid_idx], y[valid_idx]
X_test, y_test = X[test_idx], y[test_idx]

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000, seed=42).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
valid_ds = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('Train:', X_train.shape)
print('Valid:', X_valid.shape)
print('Test:', X_test.shape)


## 8. Build Number Landmark Model

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import BatchNormalization, Conv1D, Dense, Dropout, GlobalAveragePooling1D, Input, ReLU
from tensorflow.keras.optimizers import Adam

num_classes = len(selected_labels)

inputs = Input(shape=(SEQ_LEN, FEATURE_DIM), name='landmark_sequence')
x = Conv1D(96, kernel_size=3, padding='same', kernel_initializer='he_normal')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)
x = Dropout(0.15)(x)

x = Conv1D(192, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = Dropout(0.20)(x)

x = GlobalAveragePooling1D()(x)
x = Dense(192, kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = Dropout(0.25)(x)
outputs = Dense(num_classes, activation='softmax', kernel_initializer='glorot_normal', name='number_probs')(x)

model = Model(inputs=inputs, outputs=outputs, name='smart_sign_number_landmark_model')
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


## 9. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_model_number.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        mode='max',
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
    ),
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


## 10. Evaluate

In [ ]:
def plot_loss_acc(history, model_name='number_landmark_model'):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.title(f'Loss - {model_name}')
    plt.plot(history.history['loss'], c='red', label='train_loss')
    plt.plot(history.history['val_loss'], c='green', label='valid_loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.title(f'Accuracy - {model_name}')
    plt.plot(history.history['accuracy'], c='red', label='train_accuracy')
    plt.plot(history.history['val_accuracy'], c='green', label='valid_accuracy')
    plt.legend()
    plt.show()

plot_loss_acc(history)
test_metrics = model.evaluate(test_ds, return_dict=True)
test_metrics


## 11. Quick Prediction Check

In [ ]:
sample_count = min(20, len(X_test))
sample_indices = np.random.choice(len(X_test), sample_count, replace=False)
pred_probs = model.predict(X_test[sample_indices])
pred_ids = pred_probs.argmax(axis=-1)

for i, sample_index in enumerate(sample_indices):
    true_label = index_to_label[int(y_test[sample_index])]
    pred_label = index_to_label[int(pred_ids[i])]
    confidence = float(pred_probs[i, pred_ids[i]])
    print(f'True: {true_label} | Pred: {pred_label} | confidence={confidence:.3f}')


## 12. Save Model and Mapping

In [ ]:
PROJECT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(OUTPUT_MODEL_PATH)

mapping = {
    'word_to_index': {label: int(index) for label, index in label_to_index.items()},
    'index_to_word': {str(index): label for index, label in index_to_label.items()},
    'selected_words': selected_labels,
    'num_classes': int(num_classes),
    'seq_len': int(SEQ_LEN),
    'feature_dim': int(FEATURE_DIM),
    'normalized': True,
    'architecture': 'asl_number_landmark_conv1d',
    'source_dataset': str(NUMBER_DATASET_DIR),
    'test_metrics': {key: float(value) for key, value in test_metrics.items()},
}

with OUTPUT_MAPPING_PATH.open('w', encoding='utf-8') as file:
    json.dump(mapping, file, ensure_ascii=False, indent=2)

print('Saved model:', OUTPUT_MODEL_PATH)
print('Saved mapping:', OUTPUT_MAPPING_PATH)
print(json.dumps(mapping, ensure_ascii=False, indent=2))


## 13. Optional: Use This Model as Current Alnum Model

Nếu muốn dùng model số này ngay trong mode `Alphabet/Number` hiện có, đổi output thành:

```python
OUTPUT_MODEL_PATH = PROJECT_MODEL_DIR / 'model_alnum.keras'
OUTPUT_MAPPING_PATH = PROJECT_MODEL_DIR / 'mapping_alnum.json'
```

Cách đó sẽ thay model alnum hiện tại bằng model chỉ nhận diện số. Nếu muốn giữ cả chữ cái và số, nên dùng notebook `train_alphabet_landmark_model.ipynb` với dataset số được cấu hình vào `ASL_DIGITS_DIR`.